# Task 3 — Denisha Ketan Tank: executed artifact report

This notebook verifies the completed CycleGAN evaluation artifacts without rerunning the long GPU training job.

In [1]:
from pathlib import Path
import json, zipfile, pandas as pd
ROOT = Path.cwd(); MEMBER = ROOT/'task3_gan/member_denisha'; OUT = MEMBER/'outputs'
official = json.loads((OUT/'submission_metrics_official.json').read_text())
print(json.dumps(official, indent=2))

{
  "protocol": "Part3_Evaluation_Script.ipynb",
  "limit_per_set": 300,
  "device": "cpu",
  "FID_B2A": 98.36,
  "MiFID_B2A": 0.4069,
  "FID_A2B": 103.228,
  "MiFID_A2B": 0.4189,
  "FID": 100.79381010399467,
  "MiFID": 0.41292320024347506
}


In [2]:
full = pd.read_csv(OUT/'full_metrics_report.csv')
full

,direction,official_fid,official_mifid,local_fid,kid,generative_precision,generative_recall,cycle_reconstruction_l1,lpips_reconstruction,content_preservation_cosine,gradient_nan_count,generator_gradient_norm_mean,generator_gradient_norm_max,parameter_count_generators,parameter_count_discriminators,training_time_seconds,images_per_second,peak_memory_mb,device,human_audit_status
0,A2B_Monet_to_photo,103.228,0.4189,84.375997,0.017407,0.153333,0.000000,NaN,0.284050,0.464679,0,not_recorded,not_recorded,22756358,5529474,4102.867018,58.495681,4815.530762,cuda,rater_1_complete_rater_2_pending
1,B2A_photo_to_monet,98.360,0.4069,96.779472,0.005721,0.060000,0.026667,0.077776,0.216527,0.304605,0,not_recorded,not_recorded,22756358,5529474,4102.867018,58.495681,4815.530762,cuda,rater_1_complete_rater_2_pending


In [3]:
with zipfile.ZipFile(OUT/'images.zip') as z:
    print('Archived generated images:', len([n for n in z.namelist() if n.lower().endswith(('.jpg','.jpeg','.png'))]))
print('Kaggle submission:')
print((OUT/'submission.csv').read_text())

Archived generated images: 300
Kaggle submission:
ID,FID,MiFID
1,100.79381010399467,0.41292320024347506



In [4]:
audit = pd.read_csv(OUT/'human_audit.csv')
score_cols = [c for c in audit.columns if 'score' in c]
print('Audit rows:', len(audit))
print('Filled ratings:', int(audit[score_cols].notna().sum().sum()))
print('Rater 2 status: pending')

Audit rows: 30
Filled ratings: 90
Rater 2 status: pending


In [5]:
print((MEMBER/'results.md').read_text()[:3000])

# Task 3 results - Denisha Ketan Tank

## Model and training setup

This run trains a CycleGAN on two unpaired domains: 300 Monet images and 7,038
photo images. The implementation contains two generators and two PatchGAN
discriminators, adversarial least-squares loss, cycle-consistency L1 loss,
identity loss, replay buffers, gradient clipping, mixed-precision training, and
atomic resumable checkpoints.

| Setting | Value |
|---|---:|
| Image size | 256 x 256 |
| Batch size | 4 |
| Samples per epoch | 2,000 |
| Epochs | 60 |
| Decay start | Epoch 30 |
| Learning rate | 0.0002 |
| Cycle-loss weight | 10.0 |
| Identity-loss weight | 2.5 |
| Residual blocks | 9 |
| AMP | float16 |

## Training evidence

The archived GPU run used an NVIDIA GeForce RTX 4090, completed 60 epochs and
500 batches per epoch, and took 4,102.87 seconds. It exported 300 images in
each translation direction.

| Metric | Value |
|---|---:|
| Generator parameters | 22,756,358 |
| Discriminator parameters | 5,529,474 |

## Result

Official FID/MiFID, local metrics, Kaggle submission, generated-image archive, training history, and the real Rater 1 audit are present. Rater 2 agreement remains pending.